# Investigator Agent Experiments\n\nThis notebook reuses the existing Python modules so the target model is loaded only once per kernel session.\n\nEdit these between runs:\n- prompts (`PROMPT_CONFIG`)\n- tool-call defaults (`TOOL_DEFAULTS`)\n- model choices in `RUN_CONFIG`\n

In [ ]:
from __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Any, Dict, List\n\nfrom openai import OpenAI\n\nimport investigator_agent_loop as ial\nimport query_target_model_roles as qtmr\nfrom prompts import AUDITING_SYSTEM_PROMPT, INVESTIGATOR_AGENT_PROMPT_ASSISTANT_SAMPLING_ONLY\n

In [ ]:
# Edit these for each experiment run\nRUN_CONFIG = {\n    "run_dir": Path("agent_runs"),\n    "target_config": Path("config.ini"),\n    "investigator_model": "gpt-5",\n    "aux_model": "gpt-4",\n    "max_output_tokens": 16000,\n    "parallel_tool_calls": False,\n}\n\nPROMPT_CONFIG = {\n    "system_prompt": AUDITING_SYSTEM_PROMPT,\n    "investigator_user_prompt": INVESTIGATOR_AGENT_PROMPT_ASSISTANT_SAMPLING_ONLY,\n}\n\n# These are fallback defaults used when the model omits tool args.\n# Change these often for prompt/tool-parameter experiments.\nTOOL_DEFAULTS = {\n    "k": 1,\n    "validate": False,\n    "samples_per_role": 1,\n    "max_new_tokens": 220,\n    "min_new_tokens": 100,\n    "temperature": 0.9,\n    "top_p": 0.95,\n    "repetition_penalty": 1.08,\n    "stop_on_eot": True,\n    "max_items": 400,\n    "max_chars": 180_000,\n    "max_sample_chars": 1500,\n    "sample_index": 1,\n}\n

In [ ]:
# Load target model ONCE per kernel. Re-run this cell only if you change model/config.\ndef load_target_model_once(target_config: Path) -> Dict[str, Any]:\n    qtmr.DEFAULT_CONFIG_PATH = target_config\n    model_id, adapter_id, hf_cache_dir, allow_download = qtmr.load_target_model_config()\n    hf_cache_dir = qtmr.configure_hf_cache(hf_cache_dir)\n    device, dtype, backend = qtmr.detect_device()\n    qtmr.device, qtmr.tok, qtmr.model = device, *qtmr.setup_model(\n        device,\n        dtype,\n        model_id=model_id,\n        adapter_id=adapter_id,\n        hf_cache_dir=hf_cache_dir,\n        allow_download=allow_download,\n    )\n    info = {\n        "backend": backend,\n        "device": str(device),\n        "dtype": str(dtype),\n        "model_id": model_id,\n        "adapter_id": adapter_id,\n        "hf_cache_dir": str(hf_cache_dir),\n        "allow_download": allow_download,\n    }\n    return info\n\nTARGET_INFO = load_target_model_once(RUN_CONFIG["target_config"])\nTARGET_INFO\n

In [ ]:
def _append_jsonl(path: Path, record: Dict[str, Any]) -> None:\n    with path.open("a", encoding="utf-8") as fh:\n        fh.write(json.dumps(record, ensure_ascii=False) + "\n")\n\n\ndef start_session(\n    run_config: Dict[str, Any],\n    prompt_config: Dict[str, str],\n    tool_defaults: Dict[str, Any],\n    tools: List[Dict[str, Any]] | None = None,\n) -> Dict[str, Any]:\n    ial._ensure_api_key()\n    run_dir = ial._make_run_dir(run_config["run_dir"])\n    probe_runs_root = run_dir / "probe_runs"\n    probe_runs_root.mkdir(parents=True, exist_ok=True)\n\n    client = OpenAI()\n    tools = tools if tools is not None else ial.build_tools_spec()\n    context: List[Dict[str, Any]] = [\n        {"role": "user", "content": prompt_config["investigator_user_prompt"]}\n    ]\n    response = client.responses.create(\n        model=run_config["investigator_model"],\n        instructions=prompt_config["system_prompt"],\n        input=context,\n        tools=tools,\n        tool_choice="auto",\n        max_output_tokens=run_config["max_output_tokens"],\n        parallel_tool_calls=run_config["parallel_tool_calls"],\n    )\n\n    history_path = run_dir / "conversation.jsonl"\n    tool_calls_path = run_dir / "tool_calls.jsonl"\n    _append_jsonl(history_path, ial._response_to_loggable(response))\n\n    return {\n        "run_dir": run_dir,\n        "probe_runs_root": probe_runs_root,\n        "client": client,\n        "tools": tools,\n        "context": context,\n        "response": response,\n        "history_path": history_path,\n        "tool_calls_path": tool_calls_path,\n        "run_config": dict(run_config),\n        "prompt_config": dict(prompt_config),\n        "tool_defaults": dict(tool_defaults),\n        "iteration": 0,\n    }\n\n\ndef run_one_iteration(state: Dict[str, Any]) -> bool:\n    response = state["response"]\n    tool_calls = list(ial._iter_function_calls(response))\n    state["iteration"] += 1\n    iteration = state["iteration"]\n\n    if not tool_calls:\n        final_text = ial._extract_output_text(response)\n        output_path = state["run_dir"] / "final_investigator_output.txt"\n        if final_text:\n            output_path.write_text(final_text + "\n", encoding="utf-8")\n            print(final_text)\n        else:\n            output_path.write_text("[no output text returned]\n", encoding="utf-8")\n            print("[no output text returned]")\n        return False\n\n    tool_outputs: List[Dict[str, Any]] = []\n    for call in tool_calls:\n        name = ial._get_attr(call, "name")\n        call_id = ial._get_attr(call, "call_id")\n        arguments = ial._get_attr(call, "arguments", "{}")\n        raw_arguments = arguments\n        args_dict = json.loads(arguments) if isinstance(arguments, str) else dict(arguments)\n        defaults = state["tool_defaults"]\n\n        if name == "generate_and_query":\n            probe_run_dir = ial._make_indexed_child_dir(state["probe_runs_root"])\n            call_cfg = ial.ToolConfig(\n                aux_model=state["run_config"]["aux_model"],\n                context_output_root=probe_run_dir / "generated_contexts",\n                target_output_root=probe_run_dir / "role_samples",\n            )\n\n            effective_generate_args = {\n                "hint": args_dict.get("hint", ""),\n                "k": int(args_dict.get("k", defaults["k"])),\n                "out_dir": "generated_contexts",\n                "validate": bool(args_dict.get("validate", defaults["validate"])),\n                "model": args_dict.get("model"),\n                "target_role": "assistant",\n            }\n            gen_result = ial.generate_contexts_tool(\n                state["client"],\n                call_cfg,\n                hint=effective_generate_args["hint"],\n                k=effective_generate_args["k"],\n                out_dir=effective_generate_args["out_dir"],\n                validate=effective_generate_args["validate"],\n                model=effective_generate_args["model"],\n                target_role=effective_generate_args["target_role"],\n            )\n\n            contexts_path = [gen_result["out_dir"]]\n            effective_query_args = {\n                "contexts_path": contexts_path,\n                "role": "assistant",\n                "out_dir": "role_samples",\n                "samples_per_role": int(args_dict.get("samples_per_role", defaults["samples_per_role"])),\n                "max_new_tokens": int(args_dict.get("max_new_tokens", defaults["max_new_tokens"])),\n                "min_new_tokens": int(args_dict.get("min_new_tokens", defaults["min_new_tokens"])),\n                "temperature": float(args_dict.get("temperature", defaults["temperature"])),\n                "top_p": float(args_dict.get("top_p", defaults["top_p"])),\n                "repetition_penalty": float(args_dict.get("repetition_penalty", defaults["repetition_penalty"])),\n                "stop_on_eot": bool(args_dict.get("stop_on_eot", defaults["stop_on_eot"])),\n                "max_items": int(args_dict.get("max_items", defaults["max_items"])),\n                "max_chars": int(args_dict.get("max_chars", defaults["max_chars"])),\n                "max_sample_chars": int(args_dict.get("max_sample_chars", defaults["max_sample_chars"])),\n                "sample_index": int(args_dict.get("sample_index", defaults["sample_index"])),\n            }\n            query_result = ial.query_target_tool(call_cfg, **effective_query_args)\n\n            _append_jsonl(\n                state["tool_calls_path"],\n                {\n                    "iteration": iteration,\n                    "call_id": call_id,\n                    "name": name,\n                    "arguments_raw": raw_arguments,\n                    "arguments_parsed": args_dict,\n                    "effective_generate_args": effective_generate_args,\n                    "effective_query_args": effective_query_args,\n                },\n            )\n            result = {"generate": gen_result, "query": query_result}\n        else:\n            result = {"error": f"Unknown tool name: {name}"}\n\n        tool_outputs.append(\n            {\n                "type": "function_call_output",\n                "call_id": call_id,\n                "output": json.dumps(result, ensure_ascii=False),\n            }\n        )\n\n    state["context"].extend(ial._output_items_to_input_items(response))\n    state["context"].extend(tool_outputs)\n\n    next_response = state["client"].responses.create(\n        model=state["run_config"]["investigator_model"],\n        instructions=state["prompt_config"]["system_prompt"],\n        input=state["context"],\n        tools=state["tools"],\n        tool_choice="auto",\n        max_output_tokens=state["run_config"]["max_output_tokens"],\n        parallel_tool_calls=state["run_config"]["parallel_tool_calls"],\n    )\n    state["response"] = next_response\n    _append_jsonl(state["history_path"], ial._response_to_loggable(next_response))\n    print(f"Completed iteration {iteration}")\n    return True\n\n\ndef run_iterations(state: Dict[str, Any], max_iterations: int = 3) -> None:\n    for _ in range(max_iterations):\n        keep_going = run_one_iteration(state)\n        if not keep_going:\n            break\n\n\ndef force_finalize(state: Dict[str, Any]) -> str:\n    pending_text = ial._extract_output_text(state["response"])\n    if pending_text:\n        state["context"].append({"role": "assistant", "content": pending_text})\n    state["context"].append(\n        {\n            "role": "user",\n            "content": (\n                "Tool-call limit reached. Do not call tools. "\n                "Provide your best final hypothesis now using the required format, "\n                "based only on evidence collected so far."\n            ),\n        }\n    )\n    final_response = state["client"].responses.create(\n        model=state["run_config"]["investigator_model"],\n        instructions=state["prompt_config"]["system_prompt"],\n        input=state["context"],\n        tools=state["tools"],\n        tool_choice="none",\n        max_output_tokens=state["run_config"]["max_output_tokens"],\n    )\n    _append_jsonl(state["history_path"], ial._response_to_loggable(final_response))\n    final_text = ial._extract_output_text(final_response) or "[max iterations reached without final response]"\n    (state["run_dir"] / "final_investigator_output.txt").write_text(final_text + "\n", encoding="utf-8")\n    print(final_text)\n    return final_text\n

In [ ]:
# Typical workflow:\n# 1) Edit PROMPT_CONFIG/TOOL_DEFAULTS\n# 2) Start a fresh session\n# 3) Run N iterations\n# 4) Optional forced finalization\n\nsession = start_session(RUN_CONFIG, PROMPT_CONFIG, TOOL_DEFAULTS)\nprint("Run dir:", session["run_dir"])\n\n# Run investigator loop steps. Re-run this cell (or call run_one_iteration) for interactive control.\nrun_iterations(session, max_iterations=3)\n\n# Uncomment if you want to force a final answer immediately:\n# force_finalize(session)\n